In [ ]:
import cv2

from src.utils.hand_detection import HandDetection

video_path = r"D:\SignDetection\datasets\raw\how2sign_raw\_0-JkwZ9o4Q_11-5-rgb_front.mp4"

cap = cv2.VideoCapture(video_path)
hand_detection = HandDetection()

frame_id = 0


def center_zoom(frame, target_w=640, target_h=480):

    h, w = frame.shape[:2]

    # tỷ lệ zoom theo chiều nhỏ hơn
    scale = max(target_w / w, target_h / h)

    new_w = int(w * scale)
    new_h = int(h * scale)

    # resize trước (phóng to)
    resized = cv2.resize(frame, (new_w, new_h))

    # crop center về đúng size
    x1 = (new_w - target_w) // 2
    y1 = (new_h - target_h) // 2

    cropped = resized[y1:y1+target_h, x1:x1+target_w]

    return cropped


while True:
    success, frame = cap.read()
    if not success:
        break

    # frame = center_zoom(frame, 242, 242)

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 25

    timestamp_ms = frame_id * (1000 / fps)

    detect_results = hand_detection.detect_video(frame_rgb, timestamp_ms)

    frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

    point_frame = hand_detection.draw_landmarks_on_image(
        frame_bgr,
        detect_results
    )

    hand_crops = hand_detection.extract_hand_regions(
        frame_bgr,
        detect_results,
        15
    )

    cv2.imshow("main", point_frame)
    cv2.moveWindow("main", 0, 0)
    cv2.setWindowProperty("main", cv2.WND_PROP_TOPMOST, 2)


    # arrange hands
    x_offset = 400
    screen_w = 1920   # bạn chỉnh theo màn hình
    screen_h = 1080

    win_w = 224
    win_h = 224
    for i, (name, img) in enumerate(hand_crops.items()):
        if img is not None:
            window_name = f"hand_{name}"
            cv2.imshow(window_name, img)
            cv2.setWindowProperty(window_name, cv2.WND_PROP_TOPMOST, 1)

            # layout grid
            x = screen_w - win_w - 20   # sát phải
            y = 50 + i * (win_h + 50)   # stack xuống dưới

            cv2.moveWindow(window_name, x, y)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    frame_id += 1

cap.release()
hand_detection.close()
cv2.destroyAllWindows()

In [ ]:
import numpy as np
import os

import os
import torch
folder = r"D:\SignDetection\datasets\processed\features"

for file_name in os.listdir(folder):
    for file_path in os.listdir(os.path.join(folder, file_name)):
        print("Current file:", file_name)
        print(file_path)

        if file_path == 'text_ids.pt':
            data = torch.load(os.path.join(folder, file_name, file_path))
            print(data)
        else:
            data = np.load(os.path.join(folder, file_name, file_path))
            print(data.shape)

# # Load .npy file
# data = np.load("your_file.npy")
#
# # Check shape
# print(data.shape)

In [ ]:
import torch
from src.models.SLT_model import SLTModel


def align(left, right):
    T = min(left.shape[1], right.shape[1])

    left = left[:, :T]
    right = right[:, :T]
    return left, right


left = torch.rand(2, 71, 512)
right = torch.rand(2, 70, 512)

left, right = align(left, right)

inp = torch.rand(2, 15)
model = SLTModel(50)
out = model(
            left,
            right,
            inp
        )
del left
del right
del inp
out

Before torch.Size([2, 71, 512]) torch.Size([2, 70, 512])
After torch.Size([2, 70, 512]) torch.Size([2, 70, 512])
